# Quadrature comparison on flocking trajectories

This notebook applies the local quadrature implementations to a saved flocking simulation. It compares trapezoidal integration, moment matching on uniform nodes, Clenshaw–Curtis spatial nodes, and Chebyshev–Gauss spatial nodes with Fejér-I weights.

Each case uses the same particle trajectories, integration supports, library terms, and regression settings. The particle kernel is evaluated directly at each spatial rule's nodes. Time remains on the saved uniform snapshots: the three new-rule cases use degree-limited moment matching in time. All matrices use physical space/time integration units.

Run [01_flocking_discovery.ipynb](01_flocking_discovery.ipynb) first if no trajectories have been saved. Select the PySPIDER Conda interpreter.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from time import perf_counter
import sys, os, json, hashlib, uuid, subprocess
REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/PySPIDER').is_dir()), None)
if REPO is None:
    raise RuntimeError('Launch this notebook from within the PySPIDER repository.')
DYNAMICS = REPO / 'flocking/dynamics'
sys.path.insert(0, str(REPO / 'src'))
sys.path.insert(0, str(DYNAMICS))
os.environ.setdefault('NUMBA_NUM_THREADS', '2')
import numpy as np
import PySPIDER
from flocking_data import load_trajectories
from flocking_quadrature import QuadratureSRDataset
from PySPIDER.commons.library import Observable
from PySPIDER.commons.identify_models import interleave_identify
from PySPIDER.commons.sparse_reg_bf import Scaler, Initializer, ModelIterator, Residual, Threshold
assert Path(PySPIDER.__file__).resolve().is_relative_to(REPO / 'src')
print('Interpreter:', sys.executable)
print('PySPIDER:', PySPIDER.__file__)


## Data and settings

By default, use the latest saved trajectory under `runs/` with a sibling `simulation.json`. Set `SOURCE_TRAJECTORY` to select another such file; relative paths start from this notebook's directory. Existing files are read unchanged.

`MOMENT_DEGREE` limits the polynomial moments requested from the uniform-grid solver. The default 10 avoids the high-degree fit associated with using every available uniform-grid moment. It is a starting setting for a sensitivity study.


In [ ]:
SOURCE_TRAJECTORY = None
MOMENT_DEGREE = 10
DOMAIN_SEED = 1
N_DOMAINS = 12
DOMAIN_WIDTH = 24.0
DOMAIN_FRAMES = 25
CG_RES = 1.0
KERNEL_SIGMA = 4.0
WEIGHT_M = 6
WEIGHT_QMAX = 1
MAX_COMPLEXITY = 4
L_SCALE, T_SCALE = 8.0, 2.0
DISCOVERY_THRESHOLD = 2e-3
RULES = [('trapezoidal', 'trapezoidal'), ('moment-matching', 'moment-matching'),
         ('clenshaw-curtis', 'moment-matching'), ('chebyshev-gauss', 'moment-matching')]

if SOURCE_TRAJECTORY is None:
    candidates = sorted(p for p in (DYNAMICS / 'runs').glob('*/trajectories.npy') if (p.parent / 'simulation.json').is_file())
    if not candidates:
        raise FileNotFoundError('Run 01_flocking_discovery.ipynb to save a trajectory first.')
    trajectory_path = candidates[-1]
else:
    trajectory_path = Path(SOURCE_TRAJECTORY).expanduser()
    if not trajectory_path.is_absolute():
        trajectory_path = DYNAMICS / trajectory_path
trajectory_path = trajectory_path.resolve(strict=True)
simulation = json.loads((trajectory_path.parent / 'simulation.json').read_text())
trajectory_hash = hashlib.sha256(trajectory_path.read_bytes()).hexdigest()
trajectory = load_trajectories(trajectory_path, simulation_dt=simulation['dt'], box_side=simulation['L'])
positions, velocities = trajectory['positions'], trajectory['velocities']
N, _, nt = positions.shape
box_side, deltat = trajectory['box_side'], trajectory['deltat']
if nt < DOMAIN_FRAMES or DOMAIN_WIDTH + 8*KERNEL_SIGMA > box_side:
    raise ValueError('The requested padded integration domains do not fit the data.')
if MOMENT_DEGREE >= min(int(DOMAIN_WIDTH*CG_RES), DOMAIN_FRAMES):
    raise ValueError('MOMENT_DEGREE must be smaller than the sample count on each axis.')
RUN_DIR = DYNAMICS / 'runs' / ('quadrature_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ') + '_' + uuid.uuid4().hex[:8])
RUN_DIR.mkdir(parents=True, exist_ok=False)
print('Trajectory:', trajectory_path)
print('Particle array shape:', positions.shape, '; saved-frame interval:', deltat)
print('Results:', RUN_DIR)


## Evaluate libraries and run discovery

`QuadratureSRDataset` keeps the repository's discrete libraries and polynomial coarse-graining kernel. It supplies the spatial evaluation nodes and physical integration intervals for each domain. Its remaining spatial derivatives use those same coordinates. The trapezoidal case includes physical sample-spacing factors; the other rules already include interval Jacobians.

All cases use scalar/vector complexity-4 libraries with at most two coarse-grained factors per term. Derivative order on each factor is limited to one in time and two in space. Normalization scales come from the trapezoidal case and are reused across cases. Padding excludes walls from the kernel supports.

The continuity diagnostic fits the divergence coefficient with the time-derivative coefficient fixed to one. Its balance residual is the norm of the resulting column sum divided by the sum of the two term norms. This gives a common diagnostic alongside SPIDER's library-dependent training residuals.


In [ ]:
comparison_records = []
reference_columns = None
reference_domains = None
reference_scales = None
for spatial_rule, time_rule in RULES:
    started = perf_counter()
    print('\n===', spatial_rule, 'in space;', time_rule, 'in time ===')
    np.random.seed(DOMAIN_SEED)
    srd = QuadratureSRDataset(world_size=np.array([box_side, box_side, nt]),
        particle_pos=positions, data_dict={'v': velocities}, observables=[Observable(string='v', rank=1)],
        irreps=QuadratureSRDataset.all_rank2_irreps()[:2], kernel_sigma=KERNEL_SIGMA,
        cg_res=CG_RES, deltat=deltat, rho_scale=N/box_side**2,
        cache_primes=True, cleanup_cache=False, wrap=False, time_sigma=0,
        spatial_rule=spatial_rule, time_rule=time_rule, moment_degree=MOMENT_DEGREE)
    srd.make_libraries(max_complexity=MAX_COMPLEXITY, max_rho=2, max_dt=1, max_dx=2)
    for library in srd.libs.values():
        library.terms.sort(key=str)
    np.random.seed(DOMAIN_SEED)
    srd.make_domains(ndomains=N_DOMAINS, domain_size=[DOMAIN_WIDTH, DOMAIN_WIDTH, DOMAIN_FRAMES], pad=4*KERNEL_SIGMA)
    corners = [(list(d.min_corner), list(d.max_corner)) for d in srd.domains]
    if len({(tuple(a), tuple(b)) for a, b in corners}) != N_DOMAINS:
        raise ValueError('Duplicate domains: change DOMAIN_SEED or reduce N_DOMAINS.')
    srd.make_weights(m=WEIGHT_M, qmax=WEIGHT_QMAX)
    srd.set_LT_scale(L=L_SCALE, T=T_SCALE)
    srd.make_library_matrices(parallel=False)
    libraries = [srd.libs[i] for i in srd.irreps]
    columns = [[str(t) for t in lib.terms] for lib in libraries]
    if reference_domains is None:
        reference_domains, reference_columns = corners, columns
        reference_scales = [np.asarray(lib.col_weights).copy() for lib in libraries]
    else:
        assert corners == reference_domains, 'Integration supports differ between cases'
        assert columns == reference_columns, 'Library columns differ between cases'
    for lib in libraries:
        assert np.isfinite(lib.Q).all() and lib.Q.shape[0] > lib.Q.shape[1]
    assert all(np.all(np.isfinite(c)) and np.all(c > 0) for c in reference_scales)
    np.savez_compressed(RUN_DIR / (spatial_rule + '_matrices.npz'),
                        **{f'Q_{i}': lib.Q for i, lib in enumerate(libraries)},
                        **{f'col_weights_{i}': c for i, c in enumerate(reference_scales)})
    grid_record = []
    for domain in srd.domains:
        grid_record.append(dict(min_corner=[int(x) for x in domain.min_corner],
            max_corner=[int(x) for x in domain.max_corner],
            nodes=[srd._axis_coords(i, domain).tolist() for i in range(3)],
            support=[[float(domain.min_corner[i]*srd.dxs[i]), float(domain.max_corner[i]*srd.dxs[i])] for i in range(3)]))
    (RUN_DIR / (spatial_rule + '_grids.json')).write_text(json.dumps(grid_record, indent=2) + '\n')
    names = columns[0]
    a = libraries[0].Q[:, names.index('∂t ρ')]
    b = libraries[0].Q[:, names.index('∂α ρ[v_α]')]
    coefficient = -float(np.dot(b, a) / np.dot(b, b))
    balance_residual = float(np.linalg.norm(a + coefficient*b) / (np.linalg.norm(a) + abs(coefficient)*np.linalg.norm(b)))
    reg_options = []
    for lib, scales in zip(libraries, reference_scales):
        k = min(10, len(lib.terms))
        reg_options.append(dict(scaler=Scaler(sub_inds=None, char_sizes=scales, row_norms=None, unit_rows=True, train_fraction=1),
            initializer=Initializer(method='power', start_k=k),
            model_iterator=ModelIterator(max_k=k, backward_forward=True, max_passes=3),
            residual=Residual(residual_type='matrix_relative'),
            threshold=Threshold(threshold_type='jump', gamma=1.5, delta=1e-8), verbose=False))
    np.random.seed(DOMAIN_SEED)
    equations, residuals, _, _, _ = interleave_identify(libraries, reg_options,
        threshold=DISCOVERY_THRESHOLD, max_equations=12, report_accuracy=True)
    records = [dict(equation=eq.pstr(), terms=[str(t) for t in eq.terms],
                    coefficients=[float(c) for c in eq.coeffs], training_residual=float(res))
               for eq, res in zip(equations, residuals)]
    record = dict(spatial_rule=spatial_rule, time_rule=time_rule, moment_degree=MOMENT_DEGREE,
                  matrix_shapes=[list(lib.Q.shape) for lib in libraries],
                  continuity_coefficient=coefficient, continuity_balance_residual=balance_residual,
                  equations=records, seconds=perf_counter()-started)
    comparison_records.append(record)
    print('Continuity coefficient:', coefficient, '; balance residual:', balance_residual)


## Comparison and saved results

The table reports continuity fits on one trajectory and one set of integration domains. It measures sensitivity to the chosen quadrature and spatial evaluation grid. Interaction-law inference and assessment on independent trajectories remain separate steps.

`comparison.json` stores settings, coefficients, ordered term names, runtime, and source/data hashes. Each case also saves its matrices and actual domain nodes. Full Clenshaw–Curtis and Chebyshev–Gauss rules suit these freshly evaluated local grids; truncated rules are useful when retaining subsets of an existing parent grid.


In [ ]:
print(f"{'spatial rule':<20} {'time rule':<17} {'coefficient':>13} {'balance residual':>18}")
for row in comparison_records:
    print(f"{row['spatial_rule']:<20} {row['time_rule']:<17} {row['continuity_coefficient']:13.9f} {row['continuity_balance_residual']:18.8e}")
settings = {k: globals()[k] for k in ['MOMENT_DEGREE', 'DOMAIN_SEED', 'N_DOMAINS', 'DOMAIN_WIDTH',
    'DOMAIN_FRAMES', 'CG_RES', 'KERNEL_SIGMA', 'WEIGHT_M', 'WEIGHT_QMAX', 'MAX_COMPLEXITY',
    'L_SCALE', 'T_SCALE', 'DISCOVERY_THRESHOLD']}
source_paths = sorted((REPO / 'src/PySPIDER').rglob('*.py')) + [DYNAMICS / 'flocking_quadrature.py', DYNAMICS / 'flocking_data.py']
comparison = dict(created_utc=datetime.now(timezone.utc).isoformat(),
    trajectory_path=str(trajectory_path), trajectory_sha256=trajectory_hash, simulation=simulation,
    particle_count=N, snapshots_used=nt, saved_frame_dt=deltat,
    dropped_final_step=trajectory['dropped_final_step'], settings=settings,
    terms=reference_columns, cases=comparison_records, integration_measure='physical dx dy dt',
    git_head=subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO, capture_output=True, text=True, check=True).stdout.strip(),
    interpreter=sys.executable, numpy_version=np.__version__,
    source_sha256={str(p.relative_to(REPO)): hashlib.sha256(p.read_bytes()).hexdigest() for p in source_paths},
    assessment='Continuity diagnostics and discovery fits on the supplied trajectory.')
assert hashlib.sha256(trajectory_path.read_bytes()).hexdigest() == trajectory_hash
(RUN_DIR / 'comparison.json').write_text(json.dumps(comparison, indent=2, ensure_ascii=False, allow_nan=False) + '\n')
print('Saved:', RUN_DIR / 'comparison.json')
